# 03 — Test the Crop Yield Prediction Model

This notebook is only for **trying inputs and inspecting model outputs**. It does not retrain the model.

**Model being tested:** the saved log-transformed Random Forest model at `models/crop_yield_model.joblib`.

Save this notebook inside the project's `notebooks/` folder. The saved model must exist locally in `models/`; it is intentionally not tracked by Git because of its size.

## 1. Load the saved model and prediction function

Run this from the notebook saved in `notebooks/`. It loads the existing model and reuses the project's prediction function.

In [1]:
from pathlib import Path
import sys
import joblib
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MODEL_PATH = PROJECT_ROOT / "models" / "crop_yield_model.joblib"
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from crop_yield_model import predict_yield

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Saved model not found at {MODEL_PATH}. Make sure models/crop_yield_model.joblib exists."
    )

model = joblib.load(MODEL_PATH)
print("Model loaded successfully:", MODEL_PATH)

Model loaded successfully: c:\Users\rohan\OneDrive\Desktop\LAPPY\STUDY\Data Science\Crop-Yield-Analysis\models\crop_yield_model.joblib


## 2. Test one example

This uses the same illustrative example as `src/predict.py`. Input units are not confirmed in the dataset documentation, so treat this as a software/model test—not a verified real-world forecast.

In [2]:
example = pd.DataFrame([{
    "Crop": "Maize",
    "Crop_Year": 2019,
    "Season": "Kharif",
    "State": "Punjab",
    "Annual_Rainfall": 800,
    "Fertilizer": 100000,
    "Pesticide": 500,
}])

prediction = float(predict_yield(model, example)[0])
print(f"Predicted yield: {prediction:.4f}")

Predicted yield: 3.2177


## 3. Try several different inputs

These are test scenarios, not recommendations about what to plant or how much input to apply. Numerical values are illustrative.

In [3]:
test_cases = pd.DataFrame([
    {
        "Scenario": "Maize - Punjab - Kharif",
        "Crop": "Maize", "Crop_Year": 2019, "Season": "Kharif", "State": "Punjab",
        "Annual_Rainfall": 800, "Fertilizer": 100000, "Pesticide": 500,
    },
    {
        "Scenario": "Wheat - Punjab - Rabi",
        "Crop": "Wheat", "Crop_Year": 2018, "Season": "Rabi", "State": "Punjab",
        "Annual_Rainfall": 600, "Fertilizer": 90000, "Pesticide": 400,
    },
    {
        "Scenario": "Rice - Haryana - Kharif",
        "Crop": "Rice", "Crop_Year": 2019, "Season": "Kharif", "State": "Haryana",
        "Annual_Rainfall": 900, "Fertilizer": 110000, "Pesticide": 600,
    },
])

feature_columns = [
    "Crop", "Crop_Year", "Season", "State",
    "Annual_Rainfall", "Fertilizer", "Pesticide"
]
test_cases["Predicted_Yield"] = predict_yield(model, test_cases[feature_columns])
test_cases[["Scenario", "Predicted_Yield"]].round(4)

,Scenario,Predicted_Yield
0,Maize - Punjab - Kharif,3.2177
1,Wheat - Punjab - Rabi,2.3180
2,Rice - Haryana - Kharif,2.6148


## 4. Check that predictions are valid numbers

This basic check confirms the model returned finite, non-negative predictions. It does **not** prove that each prediction is accurate.

In [4]:
import numpy as np

predictions = test_cases["Predicted_Yield"].to_numpy(dtype=float)
assert np.isfinite(predictions).all(), "A prediction is not finite."
assert (predictions >= 0).all(), "A negative yield prediction was produced."
print("Basic output checks passed.")

Basic output checks passed.


## 5. Important interpretation notes

- The model predicts a number; it does not explain causes or answer general agricultural questions.
- The dataset's yield and input units have not been confirmed, so do not attach a unit to the output yet.
- Different inputs producing different predictions is a useful test, but not proof that those predictions are accurate.
- For a meaningful accuracy check, evaluate predictions against held-out records with known actual yields. That evaluation is documented in `02_data_preprocessing.ipynb` and `docs/VERSION_02_BASELINE_MODEL.md`.

In [5]:
DATA_PATH = PROJECT_ROOT / "data" / "crop_yield.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (19689, 10)


,Crop,Crop_Year,Season,State,Area,Production,Annual_Rainfall,Fertilizer,Pesticide,Yield
0,Arecanut,1997,Whole Year,Assam,73814.0,56708,2051.4,7024878.38,22882.34,0.796087
1,Arhar/Tur,1997,Kharif,Assam,6637.0,4685,2051.4,631643.29,2057.47,0.710435
2,Castor seed,1997,Kharif,Assam,796.0,22,2051.4,75755.32,246.76,0.238333
3,Coconut,1997,Whole Year,Assam,19656.0,126905000,2051.4,1870661.52,6093.36,5238.051739
4,Cotton(lint),1997,Kharif,Assam,1739.0,794,2051.4,165500.63,539.09,0.420909


In [6]:
test_df = df[df["Crop_Year"].isin([2018, 2019])].copy()

feature_columns = [
    "Crop",
    "Crop_Year",
    "Season",
    "State",
    "Annual_Rainfall",
    "Fertilizer",
    "Pesticide"
]

X_test = test_df[feature_columns]
y_actual = test_df["Yield"]

print("Records to test:", len(test_df))

Records to test: 2142


In [7]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_predicted = predict_yield(model, X_test)

results = test_df[["Crop", "Crop_Year", "State", "Yield"]].copy()
results = results.rename(columns={"Yield": "Actual_Yield"})

results["Predicted_Yield"] = y_predicted
results["Absolute_Error"] = abs(
    results["Actual_Yield"] - results["Predicted_Yield"]
)

print("MAE:", round(mean_absolute_error(y_actual, y_predicted), 4))
print("RMSE:", round(np.sqrt(mean_squared_error(y_actual, y_predicted)), 4))
print("R²:", round(r2_score(y_actual, y_predicted), 4))

results.head(10)

MAE: 71.2434
RMSE: 859.2765
R²: -0.0073


,Crop,Crop_Year,State,Actual_Yield,Predicted_Yield,Absolute_Error
5416,Arecanut,2018,Andhra Pradesh,4.153333,2.222270,1.931063
5417,Arhar/Tur,2018,Andhra Pradesh,0.311538,0.845113,0.533575
5418,Arhar/Tur,2018,Andhra Pradesh,0.310000,0.780436,0.470436
5419,Bajra,2018,Andhra Pradesh,1.300909,1.377393,0.076484
5420,Bajra,2018,Andhra Pradesh,1.353333,1.221061,0.132272
5421,Banana,2018,Andhra Pradesh,126.387692,36.145959,90.241734
5422,Cashewnut,2018,Andhra Pradesh,1.207500,0.907091,0.300409
5423,Castor seed,2018,Andhra Pradesh,0.374286,0.817815,0.443530
5424,Castor seed,2018,Andhra Pradesh,0.410000,0.643446,0.233446
5425,Coconut,2018,Andhra Pradesh,15636.632310,0.762912,15635.869398


In [8]:
results.sort_values("Absolute_Error").head(10)

,Crop,Crop_Year,State,Actual_Yield,Predicted_Yield,Absolute_Error
16745,Sannhamp,2019,Chhattisgarh,0.405714,0.405633,0.000082
5739,Other Kharif pulses,2018,Tamil Nadu,0.689524,0.689752,0.000228
16048,Masoor,2018,Bihar,0.924211,0.924462,0.000251
5797,Oilseeds total,2019,Andhra Pradesh,1.720769,1.721042,0.000273
16641,Wheat,2018,West Bengal,2.483000,2.482692,0.000308
5481,Urad,2018,Andhra Pradesh,0.782308,0.781764,0.000544
16757,Arhar/Tur,2019,Gujarat,1.122500,1.123182,0.000682
16407,Wheat,2018,Nagaland,1.830000,1.831079,0.001079
16540,Rice,2018,Uttar Pradesh,2.605600,2.607197,0.001597
5447,Moong(Green Gram),2018,Andhra Pradesh,0.678462,0.676712,0.001749


In [9]:
results.sort_values("Absolute_Error", ascending=False).head(10)

,Crop,Crop_Year,State,Actual_Yield,Predicted_Yield,Absolute_Error
5771,Coconut,2019,Andhra Pradesh,16935.058460,0.744989,16934.313471
5425,Coconut,2018,Andhra Pradesh,15636.632310,0.762912,15635.869398
16598,Coconut,2018,West Bengal,12018.407270,0.951711,12017.455559
17272,Coconut,2019,West Bengal,11918.291360,0.933145,11917.358215
5714,Coconut,2018,Tamil Nadu,11263.830000,0.789929,11263.040071
5838,Coconut,2019,Puducherry,9474.625000,0.736848,9473.888152
6063,Coconut,2019,Tamil Nadu,8795.260556,0.799651,8794.460905
5531,Coconut,2018,Assam,8664.686667,0.716273,8663.970394
5491,Coconut,2018,Puducherry,8363.155000,0.773448,8362.381552
5962,Coconut,2019,Karnataka,8344.984000,0.711803,8344.272197
